Create the client

In [2]:
from openai import OpenAI

openai_client = OpenAI()

Create a basic stream

In [3]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

stream = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    stream=True
)

response = None
for event in stream:
    if hasattr(event, 'delta'):
        print(event.delta, end='')
    if hasattr(event, 'response'):
        response = event.response
response

Once upon a time, in a faraway land called Misty Meadows, there lived a unicorn named Glitterhorn. Unlike most unicorns known for their beauty and grace, Glitterhorn was rather clumsy, with a crooked horn and a coat that seemed more dull than shiny. Despite his shortcomings, Glitterhorn was cheerful and always tried to bring joy to others.

One day, Glitterhorn decided he would throw a grand party to prove that he was just as special as the other unicorns. He invited all the creatures of the meadows, including the elegant fairies, swift-footed deer, and the wise old owl. Glitterhorn spent days preparing, gathering berries and flowers, and decorating his small clearing with bright ribbons.

The day of the party arrived. Excitement buzzed in the air, but Glitterhorn’s clumsiness soon turned the celebration into chaos. As he tried to hang a banner, he accidentally tripped over his own hooves. He tumbled into a pond, splashing water everywhere and soaking the guests. The fairies giggled, b

Response(id='resp_0c8e577e4874f1df006ab5311326d487d29fef0851fda00aa6', created_at=1790259475.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_0c8e577e4874f1df006ab531152de487d2917387188e7dbea8', content=[ResponseOutputText(annotations=[], text='Once upon a time, in a faraway land called Misty Meadows, there lived a unicorn named Glitterhorn. Unlike most unicorns known for their beauty and grace, Glitterhorn was rather clumsy, with a crooked horn and a coat that seemed more dull than shiny. Despite his shortcomings, Glitterhorn was cheerful and always tried to bring joy to others.\n\nOne day, Glitterhorn decided he would throw a grand party to prove that he was just as special as the other unicorns. He invited all the creatures of the meadows, including the elegant fairies, swift-footed deer, and the wise old owl. Glitterhorn spent days preparing, gathering berries and flowers

Ahora vamos a añadir una respuesta estructurada. Primero definiremos los modelos.

In [4]:
from pydantic import BaseModel, Field

class ArticleSection(BaseModel):
    header: str
    text: str = Field(description="text of the section in markdown")

class ArticleResponse(BaseModel):
    title: str
    subtitle: str
    sections: list[ArticleSection]

Si intentamos usar responses.parse no va a funcionar
El metodo parse no soporta streaming.

In [5]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

try:
    response = openai_client.responses.parse(
        model="gpt-4o-mini",
        input=messages,
        text_format=ArticleResponse,
        stream=True  # This doesn't work
    )
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

Error: AttributeError: 'str' object has no attribute 'output'


Para que funcione necesitamos usar responses.stream() con el parámetro text_format

In [6]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

with openai_client.responses.stream(
    model="gpt-4o-mini",
    input=messages,
    text_format=ArticleResponse,
) as stream:
    response = None
    for event in stream:
        if hasattr(event, 'delta'):
            print(event.delta, end='')
        if hasattr(event, 'response'):
            response = event.response


{"title":"The Tale of the Misguided Unicorn","subtitle":"A Story of Mistakes and Misunderstandings","sections":[{"header":"Once Upon a Time","text":"In a faraway land called Misty Meadows, a young unicorn named Sparkle dreamed of being the most magical creature in all the realms. However, Sparkle's understanding of magic was quite misguided."},{"header":"Sparkle's Ambitions","text":"Instead of using her powers for good, Sparkle believed that the more glitter and sparkles she created, the more magical she would be. So, she began to practice her magic in her forest, casting spells that filled the skies with colorful, swirling lights."},{"header":"The Chaos Unleashed","text":"One day, while trying to impress her friends, she accidentally conjured a massive rainbow tornado. It swept through Misty Meadows, turning the trees into candy canes and the rivers into chocolate streams. While it was delightful at first, the land soon became chaotic and unlivable."},{"header":"The Consequences","tex

Y despues podemos acceder al contenido del response

In [7]:
story = response.output_parsed

print('# ' + story.title)
print(story.subtitle)
print()

for section in story.sections:
    print('## ' + section.header)
    print()
    print(section.text)
    print()
    print()

# The Tale of the Misguided Unicorn
A Story of Mistakes and Misunderstandings

## Once Upon a Time

In a faraway land called Misty Meadows, a young unicorn named Sparkle dreamed of being the most magical creature in all the realms. However, Sparkle's understanding of magic was quite misguided.


## Sparkle's Ambitions

Instead of using her powers for good, Sparkle believed that the more glitter and sparkles she created, the more magical she would be. So, she began to practice her magic in her forest, casting spells that filled the skies with colorful, swirling lights.


## The Chaos Unleashed

One day, while trying to impress her friends, she accidentally conjured a massive rainbow tornado. It swept through Misty Meadows, turning the trees into candy canes and the rivers into chocolate streams. While it was delightful at first, the land soon became chaotic and unlivable.


## The Consequences

The animals who lived there were not happy with Sparkle's idea of magic. The squirrels were n

Streaming de JSON incremental
Como el backend esta generando json al especificar un esquema. Necesitamos parsear el json incrementalmente segun viene
- Para XML hay una aproximacion SAX que puede parsear documentos que estan siendo streameados sin cargar todo el documento en memoria.
- Necesitamos algo similar para json  https://github.com/alexeygrigorev/jaxn
uv add jaxn

Importamos el handler y el parser
- El parser usa un handler con callbacks para diferentes eventos
- on_field_start(path, field_name) - Called when starting to read a field
- on_field_end(path, field_name, value, parsed_value) - Called when a field is complete
- on_value_chunk(path, field_name, chunk) - Called for each character in string values
- on_array_item_end(path, field_name, item) - Called when an array item is complete

In [8]:
from jaxn import JSONParserHandler, StreamingJSONParser

Vamos a experimentar con el parser con un documento json a medias

In [9]:
raw_json = """
{"title":"The Misadventures of Sparkle","subtitle":"A Tale of Trouble","sections":[{"header":"Once Upon a Time","text":"In a magical land far away, there lived a young unicorn named Sparkle."},{"header":"The Problem","text":"Sparkle had a mischievous streak that often led to chaos."}]}
""".strip()

Creamos el handler
- Cuando termina el campo title, lo imprime como un H1
- Cuando termina el campo subtitle, lo imprime
- Cuando termina una seccion header lo imprime como un h2
- Cuando la seccion text chunks llega lo imprimer inmediatamente

In [11]:
from typing import Any, Dict

class ArticleResponseHandler(JSONParserHandler):

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '':
            if field_name == 'title':
                print(f'# {value}')
            if field_name == 'subtitle':
                print(value)
        if path == '/sections' and field_name == 'header':
            print(f'\n\n## {value}\n')

    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '/sections' and field_name == 'text':
            print(chunk, end='', flush=True)

Creamos el parser y lo testeamos

In [13]:
handler = ArticleResponseHandler()
parser = StreamingJSONParser(handler=handler)

for c in raw_json:
    parser.parse_incremental(c)

# The Misadventures of Sparkle
A Tale of Trouble


## Once Upon a Time

In a magical land far away, there lived a young unicorn named Sparkle.

## The Problem

Sparkle had a mischievous streak that often led to chaos.

Creamos ahora una función que combina la llamada al llm con streaming con la salida estructurada.
- Constuye el array de mensajes con las instrucciones y el prompt
- Crea un StreamingJSONParser con el handler anterior
- Usa responses.stream() con el parametro text_format
- Parsea cada delta incrementalmente en el handler
- Retorna la respuesta al final para acceder a los metadatos

In [15]:
def llm_structured_stream(
    user_prompt,
    output_type,
    parser_handler=JSONParserHandler(),
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    parser = StreamingJSONParser(handler=parser_handler)

    with openai_client.responses.stream(
        model="gpt-4o-mini",
        input=messages,
        text_format=output_type,
    ) as stream:
        response = None
        for event in stream:
            if hasattr(event, 'delta'):
                parser.parse_incremental(event.delta)
            if hasattr(event, 'response'):
                response = event.response

    return response

La probamos

In [16]:
instructions = "your task is to tell the user bad time stories"
user_prompt = "unicorn"

result = llm_structured_stream(
    instructions=instructions,
    user_prompt=user_prompt,
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)

# The Cursed Unicorn
A Cautionary Tale


## The Enchanted Forest

In a magical forest, there lived a unique unicorn named Luna. With shimmering silver fur and a golden horn, she was the pride of her kingdom. The forest was filled with beautiful creatures, but an ancient curse had settled over the land, leaving it in perpetual twilight.

## The Legend of the Curse

Many years ago, a greedy sorcerer sought to capture Luna for her magical powers. He cast a dark spell upon the land, ensuring that if a unicorn was ever captured, the forest would be shrouded in darkness forever. The ground became barren, flowers wilted, and the once-vibrant colors faded to gray.

## The Battle Against Darkness

Luna, brave but not foolish, decided to confront this curse. She gathered her friends, including a wise old owl and a cheeky rabbit, to create a plan. But every attempt to lift the curse only led them deeper into despair.

## The Sorcerer's Trap

One fateful night, under the dim light of the stars, th

Steaming en la documentación del RAG.
Primero creamos el codigo de nuestro RAG

In [17]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

Indexed 382 chunks from 95 documents


Ahora creamos nuestra respuesta RAG expandida con RAGResponse de la leccion anterior

In [18]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

Para el handler queremos que se haga el stream de la respuesta caracter a caracter y que muestre los metadatos al completarse
- Answer -> Lo muestra caracter a caracter inmediatamente
- Answer_Type -> Lo muestra cuando termina
- Imprime cada floowup question por cada valor que llega

In [20]:
class RAGResponseHandler(JSONParserHandler):
    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '' and field_name == 'answer':
            print(chunk, end='', flush=True)

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '' and field_name == 'answer_type':
            print('\nanswer type:', value)

    def on_array_item_end(self, path: str, field_name: str, item: Dict[str, Any] = None) -> None:
        if field_name == 'followup_questions':
            print('follow up question:', item)

Creamos ahora la función de RAG

In [21]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured_stream(
        instructions=instructions,
        user_prompt=prompt,
        output_type=RAGResponse,
        parser_handler=RAGResponseHandler()
    )

Y la testeamos

In [22]:
response = rag('llm as a judge')


In the context of using a Large Language Model (LLM) as a judge, there are two primary methods for evaluation:

1. **Reference-based Evaluation**: This method compares new responses against a set of reference responses, which is useful for regression testing or when you have a "ground truth" to compare against.
2. **Open-ended Evaluation**: This method evaluates responses based on custom criteria when no reference responses are available. 

The tutorial provides a structured approach on how to create and tune the LLM evaluator. Key steps include:
- Creating an evaluation dataset that consists of questions, target responses, new responses, and manual labels.
- Running the LLM as a judge to determine the correctness or verbosity of the new responses.
- Comparing the evaluations made by the LLM judge with manual labels to assess quality.

The tutorial also emphasizes the necessity of having basic Python knowledge and an OpenAI API key for using the LLM evaluator, and suggests using Jupyte